# Атака сапсана: та же подстановка, другая задача

*Дополнительная задача к занятию про однородные уравнения. Виджет про форму зеркала — в соседнем ноутбуке; здесь тот же приём работает на совсем непохожей задаче.*

**Физика.** У сапсана самая острая зона зрения (глубокая центральная ямка) смотрит не вперёд, а примерно под **40° вбок**. Чтобы разглядеть добычу этой зоной, летя прямо на неё, птице пришлось бы держать голову повёрнутой — а повёрнутая голова резко портит аэродинамику на скорости пикирования. Сапсан поступает иначе: голову держит прямо, а летит так, чтобы добыча всё время оставалась **под одним и тем же углом** к направлению полёта (Tucker et al., *J. Exp. Biol.*, 2000).

**Задача.** По какой траектории он тогда летит?

**Как запускать.** «Среда выполнения» → «Выполнить всё» (Ctrl+F9), затем крутить ползунки под графиком.

---

### Сценарий: нажимать по порядку

1. **«Почти прямо (5°)»** — почти прямая атака, путь длиннее прямой на 0,4 %. Так летел бы тот, у кого острое зрение смотрит вперёд.

2. **«Сапсан (40°)»** — получается спираль, и путь длиннее прямого на **31 %**. Это и есть цена того, чтобы не вертеть головой.

3. Посмотрите на отметки: **оранжевая** — направление на добычу, **чёрная** — направление полёта. Угол между ними один и тот же во всех точках, хотя расстояние до добычи меняется в сотню раз. Это и есть условие задачи, а не следствие.

4. Справа тот же угол, но **измеренный по самой нарисованной кривой** разностями, а не взятый из формулы. Прямая горизонталь = условие выполнено.

5. Переключите путь на **«проинтегрировать»**: формула спирали не используется вовсе, кривая строится из уравнения.

---

### Уравнение

Тангенс угла между радиус-вектором $(x, y)$ и касательной $(1, y')$ равен $\dfrac{x y' - y}{x + y y'}$. Приравняв его $\mathrm{tg}\,\alpha$, получаем

$$y' = \frac{x\,\mathrm{tg}\,\alpha + y}{x - y\,\mathrm{tg}\,\alpha}$$

**Однородное** — поделив числитель и знаменатель на $x$, получаем функцию только от $v = y/x$. И это опять видно **до выкладок**: в условии задачи нет ни одной длины, только угол. Значит растяжение относительно добычи обязано переводить решения в решения.

Подстановка $y = vx$ — та же, что в задаче о зеркале:

$$x\,v' = \frac{\mathrm{tg}\,\alpha\,(1+v^2)}{1 - v\,\mathrm{tg}\,\alpha} \quad\Longrightarrow\quad \ln r = \theta\,\mathrm{ctg}\,\alpha + \mathrm{const} \quad\Longrightarrow\quad r = r_0\,e^{\theta\,\mathrm{ctg}\,\alpha}$$

Логарифмическая спираль. Логарифм здесь по той же причине, что и у зеркала: своей длины у задачи нет, поэтому в ответ могут входить только отношения.

---

### Два факта, которые стоит проговорить

**Длина пути ровно $r_0/\cos\alpha$.** Потому что $ds = dr/\cos\alpha$ — угол между путём и радиусом постоянен, значит постоянна и доля продвижения «к добыче». Отношение к прямой равно $1/\cos\alpha$ и **не зависит от $r_0$**: далёкая атака и близкая проигрывают в длине одинаково.

**Геометрически сокол не долетает никогда.** Радиус обращается в ноль лишь при $\theta \to -\infty$, то есть витков бесконечно много — но длина пути при этом конечна. Бесконечное наматывание и конечный путь уживаются без противоречия.

In [ ]:
# =============================================================
#  АТАКА САПСАНА — движок: спираль, ОДУ, измерения.
#  Интерфейс — в следующей ячейке. Здесь ничего не рисуется.
# =============================================================
import numpy as np
from scipy.integrate import solve_ivp

R0 = 1.0                       # с какого расстояния начинается атака
THETA0 = np.radians(135.0)     # где сокол в начале (добыча в начале координат)
R_MIN = 0.01 * R0              # до какого сближения рисуем путь

SRC_FORMULA, SRC_ODE = 0, 1


def t_alpha(alpha_deg):
    return np.tan(np.radians(alpha_deg))


def cot_alpha(alpha_deg):
    return 1.0 / np.tan(np.radians(alpha_deg))


# ---------------- уравнение пути ----------------
def slope_dir(x, y, alpha_deg):
    """Направление касательной, НЕ деля ни на что.

    Условие задачи: добыча всё время видна под одним и тем же углом alpha
    к направлению полёта. Через тангенс угла между радиус-вектором (x, y)
    и касательной (1, y') это даёт
        (x y' - y) / (x + y y') = tg(alpha),
    откуда
        y' = (x tg(alpha) + y) / (x - y tg(alpha)).
    Уравнение ОДНОРОДНОЕ: поделив числитель и знаменатель на x, получаем
    функцию только от v = y/x. Заранее ясно почему: в условии задачи нет
    ни одной длины — только угол.

    Возвращаем вектор (dx, dy), пропорциональный касательной: так нет
    особенности там, где касательная вертикальна (x = y tg alpha)."""
    T = t_alpha(alpha_deg)
    return x - T * y, T * x + y


def slope(x, y, alpha_deg):
    dx, dy = slope_dir(x, y, alpha_deg)
    return dy / dx


# ---------------- путь: по формуле ----------------
def spiral(alpha_deg, r0=R0, r_min=R_MIN, n=2000, theta0=THETA0):
    """Логарифмическая спираль r = r0 exp((theta - theta0) ctg alpha).

    Получается из того же уравнения подстановкой y = v x:
        x v' = tg(a) (1 + v^2) / (1 - tg(a) v),
    после разделения переменных и перехода к полярным координатам
        ln r = theta ctg(alpha) + const.
    Сокол летит в сторону УБЫВАНИЯ r, то есть theta уменьшается."""
    c = cot_alpha(alpha_deg)
    dtheta = np.log(r0 / r_min) / c           # на столько повернётся радиус
    th = np.linspace(theta0, theta0 - dtheta, n)
    r = r0 * np.exp((th - theta0) * c)
    return r * np.cos(th), r * np.sin(th), th, r


# ---------------- путь: интегрированием ----------------
def path_ode(alpha_deg, r0=R0, r_min=R_MIN, n=2000, theta0=THETA0, rtol=1e-10):
    """Тот же путь, но построенный интегрированием, без формулы спирали.

    Чтобы не спотыкаться о вертикальные касательные, уравнение берётся не
    в виде dy/dx, а как автономная система с тем же полем направлений:
        dx/dt = -(x - y tg a),   dy/dt = -(x tg a + y)
    Знак минус означает «внутрь», к добыче. Правые части линейны — это
    поворот на alpha, домноженный на растяжение; отсюда и спираль."""
    T = t_alpha(alpha_deg)
    s0 = [r0 * np.cos(theta0), r0 * np.sin(theta0)]

    def rhs(t, s):
        return [-(s[0] - T * s[1]), -(T * s[0] + s[1])]

    def reached(t, s):
        return np.hypot(s[0], s[1]) - r_min
    reached.terminal = True
    reached.direction = -1

    sol = solve_ivp(rhs, [0.0, 400.0], s0, events=reached, dense_output=True,
                    rtol=rtol, atol=1e-13)
    t_end = sol.t_events[0][0] if len(sol.t_events[0]) else sol.t[-1]
    tt = np.linspace(0.0, t_end, n)
    xy = sol.sol(tt)
    return xy[0], xy[1]


def path(alpha_deg, source, **kw):
    if source == SRC_FORMULA:
        x, y, _, _ = spiral(alpha_deg, **kw)
        return x, y
    return path_ode(alpha_deg, **kw)


# ---------------- измерения ----------------
def seen_angle(x, y):
    """Под каким углом сокол видит добычу — измеряем ПО САМОМУ ПУТИ, без
    всякой формулы: угол между скоростью и направлением НА добычу.

    Скорость берём разностями по нарисованной кривой, направление на добычу
    это (-x, -y), потому что добыча в начале координат. Отсюда знак минус
    в скалярном произведении: с (+x, +y) получился бы дополнительный угол
    180° - alpha. Путь предполагается пройденным внутрь, к добыче, — так его
    и строят spiral() и path_ode()."""
    dx = np.gradient(x)
    dy = np.gradient(y)
    cross = np.abs(x * dy - y * dx)
    dot = -(x * dx + y * dy)
    return np.degrees(np.arctan2(cross, dot))


def path_length(alpha_deg, r0=R0, r_min=0.0):
    """Точная длина: ds = dr / cos(alpha), значит L = (r0 - r_min)/cos(alpha).
    Путь длиннее прямой ровно в 1/cos(alpha) раз — и это не зависит от r0."""
    return (r0 - r_min) / np.cos(np.radians(alpha_deg))


def sweep_degrees(alpha_deg, factor=100.0):
    """На сколько повернётся радиус-вектор, пока расстояние упадёт в factor раз."""
    return np.degrees(np.log(factor) / cot_alpha(alpha_deg))


print('движок загружен: добыча в начале координат, атака с r = %.0f; '
      'путь длиннее прямой в 1/cos(alpha) раз' % R0)

In [ ]:
# =============================================================
#  АТАКА САПСАНА — интерфейс.
#  Требует предыдущую ячейку (движок). Запускать после неё.
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

A_LIM = (3.0, 87.0)         # диапазон угла зрения, градусы
CONTINUOUS = True           # False -- если на слабой машине ползунки дёргаются
N_PATH = 2000               # точек в пути
PAD = 1.22                  # запас кадра вокруг пути
MARK_FRAC = 0.13            # длина отметок угла в долях полукадра
ISO_V = (-1.7, -0.5, 0.5, 1.7)

_SL = dict(continuous_update=CONTINUOUS, style={'description_width': '150px'},
           layout=widgets.Layout(width='340px'))

_EQ = (r"$\dfrac{dy}{dx} = \dfrac{x\,\mathrm{tg}\,\alpha + y}"
       r"{x - y\,\mathrm{tg}\,\alpha}$")
_EQ_R = r"$r = r_0\,e^{\theta\,\mathrm{ctg}\,\alpha}$"


class FalconApp:

    def __init__(self):
        self._busy = False
        self._legend_key = None
        self._build_controls()
        self._build_figure()
        self._wire()
        display(self.panel, self.out, self.info)
        self._refresh()

    # ---------------- органы управления ----------------
    def _build_controls(self):
        self.w_src = widgets.ToggleButtons(
            options=[('по формуле', SRC_FORMULA), ('проинтегрировать', SRC_ODE)],
            value=SRC_FORMULA, description='Путь:',
            style={'description_width': '60px', 'button_width': '155px'})
        self.w_a = widgets.FloatSlider(value=40.0, min=A_LIM[0], max=A_LIM[1],
                                       step=1.0, readout_format='.0f',
                                       description='угол зрения α, град:', **_SL)
        self.w_marks = widgets.IntSlider(value=5, min=0, max=10,
                                         description='отметок угла:', **_SL)
        self.w_field = widgets.Checkbox(value=False, indent=False,
                                        description='поле направлений')
        self.w_family = widgets.Checkbox(value=False, indent=False,
                                         description='семейство решений')
        self.b_falcon = widgets.Button(description='Сапсан (40°)',
                                       layout=widgets.Layout(width='140px'))
        self.b_straight = widgets.Button(description='Почти прямо (5°)',
                                         layout=widgets.Layout(width='160px'))
        self.b_reset = widgets.Button(description='Сброс',
                                      layout=widgets.Layout(width='90px'))
        self.out = widgets.Output()
        self.info = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Атака сапсана: держать "
                         "добычу под постоянным углом</h3>"),
            self.w_src,
            widgets.HBox([self.w_a, self.w_marks]),
            widgets.HBox([self.b_falcon, self.b_straight, self.b_reset,
                          self.w_field, self.w_family]),
        ])

    def _wire(self):
        for w in (self.w_src, self.w_a, self.w_marks, self.w_field, self.w_family):
            w.observe(self._on_change, names='value')
        self.b_falcon.on_click(lambda _b: self._set(w_a=40.0))
        self.b_straight.on_click(lambda _b: self._set(w_a=5.0))
        self.b_reset.on_click(lambda _b: self._set(
            w_src=SRC_FORMULA, w_a=40.0, w_marks=5))

    # ---------------- фигура строится ОДИН раз ----------------
    def _build_figure(self):
        self.fig, (self.ax, self.ax2) = plt.subplots(
            1, 2, figsize=(11.2, 4.9), dpi=88,
            gridspec_kw={'width_ratios': [2.35, 1.0]})
        self.fig.subplots_adjust(left=0.055, right=0.985, top=0.90,
                                 bottom=0.115, wspace=0.30)
        plt.close(self.fig)          # чтобы фигура не дублировалась под ячейкой

        ax = self.ax
        self.a_field = LineCollection([], colors='#a8a8a8', linewidths=1.0,
                                      zorder=1)
        ax.add_collection(self.a_field)
        self.a_iso = LineCollection([], colors='#8c6fc4', linewidths=1.0,
                                    linestyles=':', alpha=0.75, zorder=1)
        ax.add_collection(self.a_iso)
        self.a_iso_mark = LineCollection([], colors='#6f4fb0', linewidths=2.0,
                                         zorder=3)
        ax.add_collection(self.a_iso_mark)
        self.a_family = LineCollection([], colors='#7fbf7f', linewidths=1.2,
                                       linestyles='--', zorder=2)
        ax.add_collection(self.a_family)
        self.a_sight = LineCollection([], colors='#c46f00', linewidths=1.8,
                                      zorder=9)
        ax.add_collection(self.a_sight)
        # касательную красим в чёрный: синим она сливалась бы с самим путём
        self.a_tang = LineCollection([], colors='black', linewidths=2.2,
                                     zorder=9)
        ax.add_collection(self.a_tang)
        self.a_straight, = ax.plot([], [], color='#999999', lw=1.4, ls='--',
                                   label='прямая атака', zorder=4)
        self.a_path, = ax.plot([], [], color='#1f4e9c', lw=2.4,
                               label='путь сокола', zorder=7)
        self.a_prey, = ax.plot([0.0], [0.0], 'o', color='crimson', ms=10,
                               label='добыча', zorder=10)
        self.a_start, = ax.plot([], [], 'o', color='#1f4e9c', ms=9,
                                mfc='white', mew=2.0, label='сокол', zorder=10)
        self.t_eq = ax.text(0.015, 0.035, '', transform=ax.transAxes,
                            fontsize=12, va='bottom', ha='left',
                            bbox=dict(fc='white', ec='#cccccc', alpha=0.92,
                                      boxstyle='round,pad=0.35'))
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_aspect('equal')
        ax.grid(True, ls='--', alpha=0.35)

        a2 = self.ax2
        a2.axhline(0.0, color='#999999', lw=1.2, ls='--',
                   label='прямая атака (0°)')
        self.a_ref = a2.axhline(40.0, color='#c46f00', lw=1.2, ls=':',
                                label='заданный α')
        self.a_psi, = a2.plot([], [], color='#1f4e9c', lw=2.2,
                              label='измерено по пути')
        a2.set_xlabel('доля пройденного пути')
        a2.set_ylabel('угол на добычу, град')
        a2.set_xlim(0.0, 1.0)
        a2.set_ylim(-5.0, 92.0)
        a2.grid(True, ls='--', alpha=0.35)
        a2.set_title('условие задачи = горизонталь', fontsize=9.5)
        a2.legend(fontsize=7.5, loc='upper left')

    # ---------------- поле направлений ----------------
    def _field_segments(self, a, box):
        x0, x1, y0, y1 = box
        X, Y = np.meshgrid(np.linspace(x0, x1, 17), np.linspace(y0, y1, 17))
        ux, uy = slope_dir(X, Y, a)
        n = np.hypot(ux, uy)
        n[n < 1e-12] = 1.0
        L = 0.028 * (x1 - x0)
        ux, uy = ux / n * L, uy / n * L
        return np.stack([np.stack([X - ux, Y - uy], axis=-1),
                         np.stack([X + ux, Y + uy], axis=-1)],
                        axis=-2).reshape(-1, 2, 2)

    def _isocline_segments(self, a, box):
        """Прямые y = v·x через ДОБЫЧУ: вдоль каждой наклон поля постоянен."""
        x0, x1, y0, y1 = box
        big = 3.0 * max(abs(x0), abs(x1), abs(y0), abs(y1)) + 1.0
        lines = [[(-big, -v * big), (big, v * big)] for v in ISO_V]
        marks, L = [], 0.035 * (x1 - x0)
        for v in ISO_V:
            ux, uy = slope_dir(1.0, v, a)
            nn = L / np.hypot(ux, uy)
            for t in np.linspace(0.25, 0.95, 3):
                for s in (t, -t):
                    px = s * 0.32 * big
                    py = v * px
                    if x0 < px < x1 and y0 < py < y1:
                        marks.append([(px - ux * nn, py - uy * nn),
                                      (px + ux * nn, py + uy * nn)])
        return lines, marks

    # ---------------- отрисовка ----------------
    def render(self):
        src, a, n_marks = self.w_src.value, self.w_a.value, self.w_marks.value

        x, y = path(a, src, n=N_PATH)
        self.a_path.set_data(x, y)
        self.a_start.set_data([x[0]], [y[0]])
        self.a_straight.set_data([x[0], 0.0], [y[0], 0.0])
        handles = [self.a_path, self.a_straight, self.a_prey, self.a_start]

        # кадр подгоняем под сам путь: при малом угле он занимает узкий сектор,
        # при большом — весь круг, и фиксированная рамка годилась бы лишь раз
        fam = ([np.column_stack(spiral(a, theta0=THETA0 + d, n=900)[:2])
                for d in (np.radians(120.0), np.radians(-120.0))]
               if self.w_family.value else [])
        xs = np.concatenate([x, [0.0]] + [c[:, 0] for c in fam])
        ys = np.concatenate([y, [0.0]] + [c[:, 1] for c in fam])
        cx = 0.5 * (xs.min() + xs.max())
        cy = 0.5 * (ys.min() + ys.max())
        half = 0.5 * max(xs.max() - xs.min(), ys.max() - ys.min()) * PAD
        box = (cx - half, cx + half, cy - half, cy + half)
        mark_len = MARK_FRAC * half

        # отметки угла: луч зрения и касательная в нескольких точках пути
        sight, tang = [], []
        if n_marks:
            r = np.hypot(x, y)
            for rk in R0 * 0.62 ** np.arange(n_marks):
                i = int(np.argmin(np.abs(r - rk)))
                i = min(max(i, 1), len(x) - 2)
                px, py = x[i], y[i]
                d = np.hypot(px, py)
                if d < 1e-9:
                    continue
                sight.append([(px, py), (px - px / d * mark_len,
                                         py - py / d * mark_len)])
                tx, ty = x[i + 1] - x[i - 1], y[i + 1] - y[i - 1]
                tn = np.hypot(tx, ty)
                tang.append([(px, py), (px + tx / tn * mark_len,
                                        py + ty / tn * mark_len)])
        self.a_sight.set_segments(sight)
        self.a_tang.set_segments(tang)

        if self.w_field.value:
            iso_lines, iso_marks = self._isocline_segments(a, box)
            self.a_field.set_segments(self._field_segments(a, box))
            self.a_iso.set_segments(iso_lines)
            self.a_iso_mark.set_segments(iso_marks)
        else:
            self.a_field.set_segments([])
            self.a_iso.set_segments([])
            self.a_iso_mark.set_segments([])
        self.a_path.set_alpha(0.35 if self.w_field.value else 1.0)

        self.a_family.set_segments(fam)
        self.ax.set_xlim(box[0], box[1])
        self.ax.set_ylim(box[2], box[3])

        self.t_eq.set_text(_EQ if src == SRC_ODE else _EQ + '\n' + _EQ_R)

        key = tuple(id(h) for h in handles)
        if key != self._legend_key:
            self.ax.legend(handles=handles, loc='upper right', fontsize=8.5)
            self._legend_key = key

        ratio = 1.0 / np.cos(np.radians(a))
        self.ax.set_title('Угол зрения %.0f° — путь длиннее прямой в %.3f раза '
                          '(на %.1f%%)' % (a, ratio, (ratio - 1) * 100),
                          fontsize=12.5, pad=8)

        # правая панель: угол на добычу, измеренный по нарисованному пути
        psi = seen_angle(x, y)
        s = np.concatenate([[0.0], np.cumsum(np.hypot(np.diff(x), np.diff(y)))])
        s = s / s[-1]
        self.a_psi.set_data(s[2:-2], psi[2:-2])
        self.a_ref.set_ydata([a, a])
        dev = float(np.max(np.abs(psi[2:-2] - a)))
        return self._info_html(src, a, ratio, dev)

    # ---------------- числа под графиком ----------------
    def _info_html(self, src, a, ratio, dev):
        sweep = sweep_degrees(a)
        rows = [
            'путь длиннее прямой в <b>%.3f</b> раза = 1/cos α' % ratio,
            'поворот до сближения в 100 раз: <b>%.0f°</b> (%.2f оборота)'
            % (sweep, sweep / 360.0),
            'ctg α = %.3f' % cot_alpha(a),
        ]
        head = ('путь задан формулой спирали' if src == SRC_FORMULA
                else 'путь построен интегрированием уравнения, формула спирали '
                     'не использована')
        check = ("Угол на добычу измерен по самой кривой (разностями), а не взят "
                 "из формулы: максимальное отклонение от заданного α — "
                 "<b>%.3f°</b>." % dev)
        tail = (
            "Уравнение однородное — правая часть зависит только от y/x. Это "
            "видно до выкладок: в условии задачи нет ни одной длины, только "
            "угол, поэтому растяжение относительно добычи обязано переводить "
            "решения в решения. Подстановка y = vx даёт "
            "x·v′ = tg α (1+v²)/(1 − v tg α), после разделения переменных — "
            "ln r = θ·ctg α + const. Логарифм здесь по той же причине, что и "
            "в задаче о зеркале: своей длины у задачи нет, входить в ответ "
            "могут только отношения.")
        note = ''
        if self.w_field.value:
            note += ("<div style='color:#6f4fb0;margin-top:4px'>Фиолетовые "
                     "пунктиры — прямые через добычу; жирные чёрточки на "
                     "каждой параллельны, то есть наклон поля зависит только "
                     "от отношения y/x. Путь сокола — одна из интегральных "
                     "кривых этого поля.</div>")
        if self.w_family.value:
            lam = 2.0
            note += ("<div style='color:#3a7a3a;margin-top:4px'>Зелёные "
                     "пунктиры — другие решения того же уравнения. У "
                     "логарифмической спирали подобие особенно наглядно: "
                     "растяжение в λ раз относительно добычи — это просто "
                     "поворот на ln λ · tg α (при λ = 2 и α = %.0f° это "
                     "%.1f°). Спираль переходит сама в себя.</div>"
                     % (a, np.degrees(np.log(lam) * t_alpha(a))))
        return ("<div style='font-size:13px;line-height:1.65;max-width:1120px'>"
                "<div>%s</div><div>%s</div><div style='margin-top:3px'>%s</div>"
                "<div style='margin-top:3px'>%s</div>%s</div>"
                % (head, ' &nbsp;•&nbsp; '.join(rows), check, tail, note))

    # ---------------- реакция на события ----------------
    def _refresh(self):
        self.info.value = self.render()
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


app = FalconApp()

---

## Вопросы к виджету

1. **Можно ли было сказать, что уравнение однородное, не выводя его?** Что именно в формулировке задачи это гарантирует — и чем это рассуждение отличается от такого же в задаче о зеркале?

2. **Почему отношение длины пути к прямой зависит только от $\alpha$**, а от начального расстояния $r_0$ — нет? Попробуйте объяснить это, не решая уравнения.

3. **Сокол наматывает бесконечно много витков, но путь конечен.** Где здесь противоречие и почему его на самом деле нет?

4. **Сравните с зеркалом.** Два совершенно непохожих условия — «все лучи в одну точку» и «добыча под постоянным углом» — дают уравнения, которые берутся одной и той же подстановкой. Что общего у этих двух задач? Загляните в обоих виджетах в поле направлений: чем похожи картинки и что играет роль особой точки в каждой?

5. **Что происходит при $\alpha \to 90^\circ$?** Поставьте ползунок на 87° и объясните и картинку, и число «путь длиннее в 19 раз».

6. **Включите «семейство решений».** Растяжение спирали относительно добычи оказывается просто её поворотом. Почему? (Подсказка: подставьте $r \to \lambda r$ в $r = r_0 e^{\theta\,\mathrm{ctg}\,\alpha}$.)

## Жёлтая рамка (необязательное, на дом)

При каком угле зрения $\alpha$ путь сапсана окажется ровно **вдвое** длиннее прямой атаки? Ответ получите формулой и проверьте ползунком.

<details>
<summary>Ответ</summary>

Из $1/\cos\alpha = 2$ сразу $\cos\alpha = 1/2$, то есть $\alpha = 60^\circ$. Ползунок подтверждает: 2,000.

Заодно видно, насколько реальные 40° — щадящий компромисс: 31 % проигрыша в длине против 100 % при 60°.
</details>

---

*Что можно править в коде:* начальное расстояние `R0`, положение сокола `THETA0` и глубина сближения `R_MIN` — в начале ячейки с движком; диапазон угла `A_LIM`, число точек пути `N_PATH` и переключатель `CONTINUOUS` — в начале ячейки с интерфейсом.